In [11]:
import sys
sys.path.append('..')
import numpy as np
import brainpy as bp
import brainpy.math as bm
import matplotlib 
import matplotlib.pyplot as plt
from CANN_DDM_model import CANN_DDM_model
from tqdm import tqdm
import pickle
import seaborn as sns
import json
import matplotlib.pyplot as plt
import os
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from scipy.stats import pearsonr, kendalltau

 run 100 trials, each with randomly selected drift rate $A \sim \text{Uniform}[-1,1]$ and fixed noise scale $\sigma_W =0.1$.



In [12]:
with open('../results/model_config_default.json') as data:
    CANN_params = json.load(data)
    decision_space_params = CANN_params['decision_space_params']
    edge_pop = CANN_params['edge_pop']
    bump_pop = CANN_params['bump_pop']


In [13]:
pos_offset = 0.003
model = CANN_DDM_model(CANN_params = CANN_params)


In [14]:
t_start = int(decision_space_params['t_start'])
k1 = edge_pop['k1']
k2 = edge_pop['k2']
K0 = edge_pop['k0']

# Aggregate over multiple runs and compute per-bin per-neuron mean firing rates
x_list = []
r_E_list = []
r_B_list = []
num_runs = 100
np.random.seed(2025)
for i in tqdm(range(num_runs)):
    drift_rate = np.random.uniform(-1, 1)
    decision_space_params.update({'seed': 2025, 'drift_rate': drift_rate})
    CANN_params.update({'decision_space_params': decision_space_params})
    model = CANN_DDM_model(CANN_params=CANN_params)
    mon_vars = ['theta_B', 'theta_E', 'hit_boundary', 'x_B', 'x_E', 'c_BE_dyn', 'r_E', 'r_B']
    runner, RT = model.run_simulation(mon_vars=mon_vars, progress_bar=False, pos_offset=pos_offset, get_RT=True)

    if not RT:
        continue
    else:
        t_end = int(t_start + RT)
        # x_E: (T,)
        x_E = runner.mon.x_E[t_start:t_end, 0]
        # r_E: (T, N)
        r_E = runner.mon.r_E[t_start:t_end, k1:k2]
        r_B = runner.mon.r_B[t_start:t_end, k1:k2]
        x_list.append(x_E)
        r_E_list.append(r_E)
        r_B_list.append(r_B)



100%|██████████| 100/100 [01:39<00:00,  1.00it/s]


In [15]:
def signed_log_transform(x, s0=1e-3):
    """Signed-log transform to stabilize DV scale."""
    x = np.asarray(x)
    return np.sign(x) * np.log1p(np.abs(x) / s0)

def build_design(R_list, x_list, s0=1e-3):
    """
    Build standardized design matrix X (sum(T_i) x N) and target y (sum(T_i),)
    from per-trial inputs. Assumes R_i shape is (T_i, N).
    """
    X_blocks, Y_blocks, trial_idx = [], [], []
    for i, (R_i, x_i) in enumerate(zip(R_list, x_list)):
        Ti, N = R_i.shape
        X_blocks.append(R_i)  # (Ti, N), already time x neurons
        x_i = np.asarray(x_i)
        if x_i.shape == (Ti,) or (x_i.ndim == 1 and x_i.size == Ti):
            y_i = x_i
        else:
            y_i = np.repeat(float(x_i.reshape(-1)[0]), Ti)
        Y_blocks.append(y_i)
        trial_idx.append(np.full(Ti, i, dtype=int))
    X = np.concatenate(X_blocks, axis=0)    # (T_sum, N)
    y = np.concatenate(Y_blocks, axis=0)    # (T_sum,)
    trial_idx = np.concatenate(trial_idx, axis=0)
    Xz = StandardScaler().fit_transform(X)
    y_log = signed_log_transform(y, s0=s0)
    return Xz, y_log, trial_idx

def grouped_folds(trial_idx, n_splits=6, random_state=0):
    """
    Create grouped folds so that samples from the same trial never split across train/val.
    Returns a list of (train_indices, val_indices).
    """
    rng = np.random.default_rng(random_state)
    trials = np.unique(trial_idx)
    rng.shuffle(trials)
    chunks = np.array_split(trials, n_splits)
    folds = []
    for k in range(n_splits):
        val_trials = set(chunks[k].tolist())
        val_mask = np.isin(trial_idx, list(val_trials))
        tr_mask = ~val_mask
        folds.append((np.where(tr_mask)[0], np.where(val_mask)[0]))
    return folds

def fit_ridge_axis_with_cv(X, y, trial_idx, alphas=np.logspace(-3, 3, 15), n_splits=6, random_state=0):
    """
    Select alpha by grouped CV, then refit on all data. Also return per-fold weights for stability.
    Returns:
      - w_unit: unit-norm weight vector (N,)
      - yhat  : predictions on all samples
      - alpha : selected alpha
      - metrics: dict with r2, r, tau
      - fold_weights: list of unit-norm weights fit on each TRAIN fold (stability analysis)
    """
    folds = grouped_folds(trial_idx, n_splits=n_splits, random_state=random_state)

    # Hyperparameter selection
    best_alpha, best_score = None, -np.inf
    for a in alphas:
        scores = []
        for tr_idx, va_idx in folds:
            mdl = Ridge(alpha=a, fit_intercept=True)
            mdl.fit(X[tr_idx], y[tr_idx])
            y_pred = mdl.predict(X[va_idx])
            scores.append(r2_score(y[va_idx], y_pred))
        score = float(np.mean(scores))
        if score > best_score:
            best_score, best_alpha = score, float(a)

    # Fit on all data
    final = Ridge(alpha=best_alpha, fit_intercept=True)
    final.fit(X, y)
    yhat = final.predict(X)
    w = final.coef_.astype(float)
    w_unit = w / (np.linalg.norm(w) + 1e-12)

    r2 = r2_score(y, yhat)
    r, _ = pearsonr(y, yhat)
    tau, _ = kendalltau(y, yhat)
    metrics = {"r2": float(r2), "r": float(r), "tau": float(tau)}

    # Fit per-fold weights on each TRAIN split (to assess stability of w)
    fold_weights = []
    for tr_idx, va_idx in folds:
        mdl = Ridge(alpha=best_alpha, fit_intercept=True)
        mdl.fit(X[tr_idx], y[tr_idx])
        w_k = mdl.coef_.astype(float)
        w_k = w_k / (np.linalg.norm(w_k) + 1e-12)
        fold_weights.append(w_k)

    return w_unit, yhat.astype(float), best_alpha, metrics, fold_weights

def cosine_similarity_matrix(W):
    """
    W: list/array of weight vectors (m x N).
    Returns cosine similarity matrix (m x m).
    """
    W = np.vstack(W)                    # (m, N)
    W = W / (np.linalg.norm(W, axis=1, keepdims=True) + 1e-12)
    return W @ W.T

# --------------------------- Plotting ---------------------------
def plot_joint_tracking(y_true, yhat_E, yhat_B, metrics_E, metrics_B, path, n_bins=25, subsample=20000):
    """
    Single figure combining Edge + Bump tracking:
      - Scatter (subsampled) of predicted vs true for both Edge and Bump
      - Binned calibration curves (mean ± 95% CI) for Edge and Bump
      - Identity line
      - Annotated R^2 and r for both
    """
    # Subsample points for readability
    n = len(y_true)
    if n > subsample:
        idx = np.random.default_rng(0).choice(n, size=subsample, replace=False)
    else:
        idx = np.arange(n)
    yt = y_true[idx]
    ye = yhat_E[idx]
    yb = yhat_B[idx]

    # Compute binned means & CIs
    def binned_curve(y_true, y_pred, nb):
        q = np.quantile(y_true, np.linspace(0, 1, nb+1))
        centers, means, lows, highs = [], [], [], []
        for i in range(nb):
            lo, hi = q[i], q[i+1]
            mask = (y_true >= lo) & (y_true <= hi) if i == nb-1 else (y_true >= lo) & (y_true < hi)
            if not np.any(mask):
                continue
            yt_bin = y_true[mask]
            yp_bin = y_pred[mask]
            centers.append(np.mean(yt_bin))
            m = float(np.mean(yp_bin))
            means.append(m)
            # 95% CI via normal approx
            se = float(np.std(yp_bin, ddof=1) / np.sqrt(len(yp_bin)))
            lows.append(m - 1.96*se)
            highs.append(m + 1.96*se)
        return np.array(centers), np.array(means), np.array(lows), np.array(highs)

    cE, mE, lE, hE = binned_curve(yt, ye, n_bins)
    cB, mB, lB, hB = binned_curve(yt, yb, n_bins)

    # Plot
    plt.figure(figsize=(7, 6))
    lims = [min(np.min(yt), np.min(ye), np.min(yb)), max(np.max(yt), np.max(ye), np.max(yb))]
    # Identity line
    plt.plot(lims, lims, 'k--', linewidth=1, label='Identity')

    # Scatter (light alpha)
    plt.scatter(yt, ye, s=8, alpha=0.25, label='Edge')
    plt.scatter(yt, yb, s=8, alpha=0.25, label='Bump')


    plt.xlabel(r"$\log \mathrm{DV}$", fontsize=14)
    plt.ylabel(r"Decoded $\mathrm{DV}$", fontsize=14)
    plt.title("Edge & Bump tracking of DV", fontsize=16)
    plt.legend(loc='best', frameon=False)
    plt.tight_layout()

    # Annotation of metrics
    # text_str = (f"Edge: R²={metrics_E['r2']:.3f}, r={metrics_E['r']:.3f}\n"
    #             f"Bump: R²={metrics_B['r2']:.3f}, r={metrics_B['r']:.3f}")
    # plt.gca().text(0.02, 0.98, text_str, transform=plt.gca().transAxes,
    #                va='top', ha='left', fontsize=10)

    plt.savefig(path, dpi=200)
    plt.close()

def plot_stability_heatmap(W_list, title, path):
    """
    Plot cosine similarity heatmap for a list of fold weight vectors.
    """
    C = cosine_similarity_matrix(W_list)
    plt.figure(figsize=(4.6, 4))
    im = plt.imshow(C, vmin=0, vmax=1, cmap='viridis', origin='lower')
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.title(title)
    plt.xlabel("Fold")
    plt.ylabel("Fold")
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()
    # Return summary stats
    # Off-diagonal similarities
    m = C.shape[0]
    off = C[~np.eye(m, dtype=bool)]
    return float(np.mean(off)), float(np.std(off))

In [16]:
# ========= Parameters =========
OUTDIR = "../figs/"
os.makedirs(OUTDIR, exist_ok=True)
S0 = 1e-3       # stabilizer for signed-log transform
N_SPLITS = 5    # number of folds for grouped cross-validation
SUBSAMPLE_SCATTER = 5000  # max points for scatter to keep the plot readable
N_BINS = 25             # bins for calibration curves

In [17]:
# --------------------------- Main ---------------------------
# 1) Build designs (expects shapes: R_i is (T_i, N))
XE, yE, trE = build_design(r_E_list, x_list, s0=S0)
XB, yB, trB = build_design(r_B_list, x_list, s0=S0)

# 2) Fit 1D axis for Edge and Bump; also get per-fold weights for stability
wE, yhatE, alphaE, metricsE, wE_folds = fit_ridge_axis_with_cv(
    XE, yE, trE, n_splits=N_SPLITS, random_state=0
)
wB, yhatB, alphaB, metricsB, wB_folds = fit_ridge_axis_with_cv(
    XB, yB, trB, n_splits=N_SPLITS, random_state=0
)




/projectnb/ecog-eeg/cyw6/.conda/envs/cann_ddm_v2/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=2.27015e-09): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/projectnb/ecog-eeg/cyw6/.conda/envs/cann_ddm_v2/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=2.69379e-09): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/projectnb/ecog-eeg/cyw6/.conda/envs/cann_ddm_v2/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=4.32487e-09): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/projectnb/ecog-eeg/cyw6/.conda/envs/cann_ddm_v2/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=2.78179e-09): result may not be accurate.
 

In [18]:
# 3) Single-figure tracking visualization (Edge + Bump together)
plot_joint_tracking(
    y_true=yE,  # same transform as yB; both derived from x_list
    yhat_E=yhatE,
    yhat_B=yhatB,
    metrics_E=metricsE,
    metrics_B=metricsB,
    path=os.path.join(OUTDIR, "tracking_joint.png"),
    n_bins=N_BINS,
    subsample=SUBSAMPLE_SCATTER
)